# Calculate climatologies for NPP in ESM4 historical simulations

In [1]:
import xarray as xr
import dask
from dask.diagnostics import ProgressBar
import gfdl_utils as gu

In [2]:
variable = "intpp"
yearstart = 1998
yearend = 2014

In [3]:
basedir_out = "/projects/SOCCOM/graemem/projects/esm4_ppe/data/processed/esm4historical"
yearstr = "_".join([str(yearstart),str(yearend)])
filename_prefix = ".".join(["ESM4_historical_D1",yearstr])
filename_suffix = ".".join([variable,"nc"])

In [4]:
# Load model data
pathDict = {"pp":"/projects/SOCCOM/datasets/archive/oar.gfdl.cmip6/ESM4/DECK/ESM4_historical_D1/gfdl.ncrc4-intel16-prod-openmp/pp",
           "add":"intpp"}
pathDict["ppname"] = gu.core.find_variable(pathDict["pp"],pathDict["add"])[0]
pathDict["out"]="ts"
pathDict["local"]=gu.core.get_local(pathDict["pp"],pathDict["ppname"],pathDict["out"])
pathDict["time"]="*"
path = gu.core.get_pathspp(**pathDict)
ds = xr.open_mfdataset(path).sel(time=slice(str(yearstart)+"-01-01",str(yearend)+"-12-31"))

In [7]:
ds = ds.chunk({'time':-1,'xh':'auto','yh':'auto'}).drop_vars('average_DT')

## Time mean

In [8]:
with dask.config.set(**{'array.slicing.split_large_chunks': False}):
    ds_climmean = ds.groupby('time.month').mean()

In [9]:
with ProgressBar():
    ds_climmean = ds_climmean.compute()

[########################################] | 100% Completed | 2.85 ss


In [10]:
filename = ".".join([filename_prefix,"climmean",filename_suffix])
outpath = "/".join([basedir_out,filename])
ds_climmean.to_netcdf(outpath)

## Anomaly variance

In [11]:
# Reload climatological mean
ds_climmean = xr.open_dataset(outpath)

In [12]:
with dask.config.set(**{'array.slicing.split_large_chunks': False}):
    ds_climanomvar = (ds.groupby('time.month') - ds_climmean).groupby('time.month').std()**2

/home/graemem/.conda/envs/core/lib/python3.10/site-packages/xarray/core/indexing.py:1446: PerformanceWarning: Slicing with an out-of-order index is generating 17 times more chunks
  return self.array[key]


In [14]:
with ProgressBar():
    ds_climanomvar = ds_climanomvar.compute()

[######                                  ] | 16% Completed | 1.19 s ms

/home/graemem/.conda/envs/core/lib/python3.10/site-packages/dask/array/numpy_compat.py:40: RuntimeWarning: invalid value encountered in true_divide
  x = np.divide(x1, x2, out)


[########################################] | 100% Completed | 5.09 s


In [15]:
filename = ".".join([filename_prefix,"climanomvar",filename_suffix])
outpath = "/".join([basedir_out,filename])
ds_climmean.to_netcdf(outpath)